# 01 - Parse Resume PDF

This notebook extracts text from every PDF in `data/`, converts each page into a document object, and saves the parsed result for the vectorization step.

In [1]:
from pathlib import Path
import json
import re


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    if (current / "data").exists():
        return current
    if (current.parent / "data").exists():
        return current.parent
    raise FileNotFoundError("Could not find the data folder from this notebook location.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PDF_PATHS = sorted(DATA_DIR.glob("*.pdf"))
PARSED_DOCS_PATH = DATA_DIR / "resume_documents.json"

print(f"Project root: {PROJECT_ROOT}")
if not PDF_PATHS:
    raise FileNotFoundError(f"No PDF files found in {DATA_DIR}")

print(f"Found {len(PDF_PATHS)} PDF file(s):")
for pdf_path in PDF_PATHS:
    print(f"- {pdf_path.name}")

Project root: /Users/atharv/projects/attem
Found 1 PDF file(s):
- resume.pdf


In [2]:
import pymupdf


def clean_text(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


documents = []

for pdf_path in PDF_PATHS:
    with pymupdf.open(pdf_path) as pdf:
        for page_index, page in enumerate(pdf, start=1):
            text = clean_text(page.get_text("text"))
            if not text:
                continue
            documents.append(
                {
                    "text": text,
                    "metadata": {
                        "source": pdf_path.name,
                        "page": page_index,
                        "parser": "pymupdf",
                    },
                }
            )

print(f"Parsed {len(documents)} document page(s).")
print(f"Total characters: {sum(len(doc['text']) for doc in documents):,}")

if not documents:
    raise ValueError("No text was extracted. If the resume is scanned, add OCR before chunking.")

Parsed 1 document page(s).
Total characters: 3,908


## Optional: pypdf fallback

Run this cell only if you want to compare PyMuPDF extraction with `pypdf`.

In [3]:
from pypdf import PdfReader


reader = PdfReader(str(PDF_PATHS[0]))
pypdf_pages = []

for page_index, page in enumerate(reader.pages, start=1):
    text = clean_text(page.extract_text() or "")
    if text:
        pypdf_pages.append({"source": PDF_PATHS[0].name, "page": page_index, "characters": len(text), "preview": text[:300]})

pypdf_pages[:3]

[{'source': 'resume.pdf',
  'page': 1,
  'characters': 3925,
  'preview': 'Atharv Tembhurnikar\nCollege Park, MD, USA\n♂phone+1 240-886-6670 ✉ attem@umd.edu /linkedinLinkedIn /githubGitHub /gl⌢bePortfolio\n Blogs /userIEEE Profile\nExperience\nT eaching Assistant (Agentic AI) – University of Maryland (UMD)Aug 2026 – Present\nMachine Learning Research Assistant – University of Ma'}]

In [4]:
for doc in documents:
    source = doc["metadata"]["source"]
    page = doc["metadata"]["page"]
    preview = doc["text"][:1_000]
    print(f"\n--- {source} | page {page} preview ---\n{preview}")


--- resume.pdf | page 1 preview ---
Atharv Tembhurnikar
College Park, MD, USA
 +1 240-886-6670
# attem@umd.edu
ï LinkedIn
§ GitHub  Portfolio
Blogs IEEE Profile
Experience
Teaching Assistant (Agentic AI) – University of Maryland (UMD)
Aug 2026 – Present
Machine Learning Research Assistant – University of Maryland (UMD)
Dec 2025 – Jul 2026
• Developed ML-based pipelines for cell-wise annotation and intensity tracking on artificial-nose sensor data, enabling
structured capture of cellular responses and temporal patterns over time.
• Engineered interpretable feature representations (response amplitude, decay dynamics, temporal slopes) to quantify cell
behavior, supporting data-driven validation of experimental hypotheses and robust pattern classification.
Data Science Intern – Hackveda Private Limited (Pune, India)
Jan 2025 – Jun 2025
• Analyzed real-time e-commerce transaction data to identify trends in orders, pricing, product categories, and regional
demand, supporting improved inve

In [5]:
PARSED_DOCS_PATH.write_text(json.dumps(documents, indent=2), encoding="utf-8")
print(f"Saved parsed documents to {PARSED_DOCS_PATH}")

Saved parsed documents to /Users/atharv/projects/attem/data/resume_documents.json
